In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
!git clone https://github.com/mateenyaqoob/unstructuredata_tfidf_repo.git

Cloning into 'unstructuredata_tfidf_repo'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 54 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 25.71 KiB | 6.43 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [ ]:
%cd /content/unstructuredata_tfidf_repo

/content/unstructuredata_tfidf_repo


In [ ]:
!ls
!ls src
!ls data

data	 notebooks  requirements.txt  task_to_do.txt
LICENSE  README.md  src		      tests
__init__.py  load_data.py  run_activity.py  tfidf_utils.py
documents.py


In [ ]:
from src.load_data import load_documents

# Load data
titles, documents = load_documents()

# Display total documents
print("Total documents loaded:", len(documents))
print("-" * 40)

# Display word count of each document
for title, doc in zip(titles, documents):
    print(f"{title}: {len(doc.split())} words")

# Verify requirements
assert len(documents) == 10, "Expected 10 documents"
assert len(titles) == 10, "Expected 10 titles"
assert all(len(doc.split()) > 30 for doc in documents), \
    "Some documents contain 30 or fewer words"

print("-" * 40)
print("All checks passed!")

Total documents loaded: 10
----------------------------------------
AI in Healthcare: 85 words
Climate Change: 83 words
Basketball Strategy: 85 words
Italian Pasta: 91 words
Space Exploration: 86 words
Cybersecurity: 88 words
Music Theory: 87 words
Gardening Vegetables: 89 words
Economics Inflation: 89 words
Public Health: 91 words
----------------------------------------
All checks passed!


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words="english")

# Fit on documents and transform text into numerical vectors
tfidf_matrix = vectorizer.fit_transform(documents)

# Print matrix shape
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

# Print vocabulary size
print("Vocabulary Size:", len(vectorizer.vocabulary_))

TF-IDF Matrix Shape: (10, 537)
Vocabulary Size: 537


In [ ]:
import nltk
from nltk import FreqDist

# Get terms and convert TF-IDF matrix to an array
terms = vectorizer.get_feature_names_out()
matrix = tfidf_matrix.toarray()

print("First 3 Documents x First 10 Terms")

for i in range(3):
    print("\nDocument:", titles[i])

    for j in range(10):
        print(terms[j], ":", round(matrix[i][j], 4))

First 3 Documents x First 10 Terms

Document: AI in Healthcare
ability : 0.0
accelerate : 0.0
access : 0.0
accountability : 0.1246
accuracy : 0.1246
accurate : 0.0
actions : 0.0
activity : 0.0
adaptation : 0.0
add : 0.0

Document: Climate Change
ability : 0.0
accelerate : 0.0
access : 0.0
accountability : 0.0
accuracy : 0.0
accurate : 0.1248
actions : 0.1248
activity : 0.0
adaptation : 0.1248
add : 0.0

Document: Basketball Strategy
ability : 0.1301
accelerate : 0.0
access : 0.0
accountability : 0.0
accuracy : 0.0
accurate : 0.0
actions : 0.0
activity : 0.0
adaptation : 0.0
add : 0.0


In [ ]:
for i, doc in enumerate(documents):

    # Get terms and their TF-IDF scores
    scores = list(zip(terms, matrix[i]))

    # Sort by highest TF-IDF score
    scores.sort(key=lambda x: x[1], reverse=True)

    print("\nDocument:", titles[i])
    print("Top 5 TF-IDF Terms:")

    for word, score in scores[:5]:
        print(word, ":", round(score, 4))


Document: AI in Healthcare
Top 5 TF-IDF Terms:
ai : 0.3738
healthcare : 0.2492
use : 0.1648
accountability : 0.1246
accuracy : 0.1246

Document: Climate Change
Top 5 TF-IDF Terms:
change : 0.2496
emissions : 0.2496
climate : 0.2122
energy : 0.2122
accurate : 0.1248

Document: Basketball Strategy
Top 5 TF-IDF Terms:
ability : 0.1301
analytics : 0.1301
basketball : 0.1301
breaks : 0.1301
chemistry : 0.1301

Document: Italian Pasta
Top 5 TF-IDF Terms:
pasta : 0.4431
sauces : 0.3323
sauce : 0.2215
different : 0.1883
add : 0.1108

Document: Space Exploration
Top 5 TF-IDF Terms:
space : 0.349
mars : 0.2327
missions : 0.2327
accelerate : 0.1163
advanced : 0.1163

Document: Cybersecurity
Top 5 TF-IDF Terms:
cybersecurity : 0.2494
systems : 0.212
data : 0.1649
attackers : 0.1247
attacks : 0.1247

Document: Music Theory
Top 5 TF-IDF Terms:
music : 0.247
musicians : 0.247
practice : 0.2099
allows : 0.1235
artists : 0.1235

Document: Gardening Vegetables
Top 5 TF-IDF Terms:
soil : 0.3581
pests : 

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

for title, doc in zip(titles, documents):
    words = word_tokenize(doc)
    print(title, ":", len(words), "tokens")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


AI in Healthcare : 101 tokens
Climate Change : 97 tokens
Basketball Strategy : 102 tokens
Italian Pasta : 108 tokens
Space Exploration : 101 tokens
Cybersecurity : 109 tokens
Music Theory : 111 tokens
Gardening Vegetables : 108 tokens
Economics Inflation : 109 tokens
Public Health : 115 tokens


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Function to search documents
def search_documents(query):

    # Convert query into TF-IDF vector
    query_vector = vectorizer.transform([query])

    # Calculate cosine similarity with all documents
    similarity_scores = cosine_similarity(
        query_vector, tfidf_matrix
    ).flatten()

    # Rank documents from highest to lowest similarity
    ranked_indices = similarity_scores.argsort()[::-1]

    print("\nQuery:", query)
    print("-" * 40)

    for i in ranked_indices:
        print(titles[i], ":", round(similarity_scores[i], 4))

    return titles[ranked_indices[0]]


# Query 1
result1 = search_documents(
    "renewable energy and climate policy"
)

# Query 2
result2 = search_documents(
    "password security and phishing"
)

# Query 3
result3 = search_documents(
    "healthy sleep exercise and diet"
)


Query: renewable energy and climate policy
----------------------------------------
Climate Change : 0.3235
Economics Inflation : 0.0952
Gardening Vegetables : 0.0485
Public Health : 0.0
Cybersecurity : 0.0
Music Theory : 0.0
Space Exploration : 0.0
Italian Pasta : 0.0
Basketball Strategy : 0.0
AI in Healthcare : 0.0

Query: password security and phishing
----------------------------------------
Cybersecurity : 0.1764
Public Health : 0.0
Economics Inflation : 0.0
Gardening Vegetables : 0.0
Music Theory : 0.0
Space Exploration : 0.0
Italian Pasta : 0.0
Basketball Strategy : 0.0
Climate Change : 0.0
AI in Healthcare : 0.0

Query: healthy sleep exercise and diet
----------------------------------------
Public Health : 0.2849
Gardening Vegetables : 0.0447
Economics Inflation : 0.0
Music Theory : 0.0
Cybersecurity : 0.0
Space Exploration : 0.0
Italian Pasta : 0.0
Basketball Strategy : 0.0
Climate Change : 0.0
AI in Healthcare : 0.0


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Original vectorizer vocabulary size
print("Original Vocabulary Size:",
      len(vectorizer.vocabulary_))

# Create new vectorizer
vectorizer_new = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2
)

# Fit and transform documents
tfidf_matrix_new = vectorizer_new.fit_transform(documents)

# Print new vocabulary size
print("New Vocabulary Size:",
      len(vectorizer_new.vocabulary_))

print("New TF-IDF Matrix Shape:",
      tfidf_matrix_new.shape)

Original Vocabulary Size: 537
New Vocabulary Size: 69
New TF-IDF Matrix Shape: (10, 69)


In [ ]:
# Get new vocabulary terms
new_terms = vectorizer_new.get_feature_names_out()

# Convert new matrix to array
new_matrix = tfidf_matrix_new.toarray()

# Print top 5 terms for each document
for i, title in enumerate(titles):

    scores = new_matrix[i]

    # Get indices of top 5 scores
    top_indices = np.argsort(scores)[::-1][:5]

    print("\nDocument:", title)
    print("Top 5 Terms:")

    for index in top_indices:
        print(new_terms[index], ":", round(scores[index], 4))


Document: AI in Healthcare
Top 5 Terms:
use : 0.3915
risk : 0.2517
privacy : 0.2517
support : 0.2517
systems : 0.2517

Document: Climate Change
Top 5 Terms:
climate : 0.3862
energy : 0.3862
trade : 0.1931
offs : 0.1931
trade offs : 0.1931

Document: Basketball Strategy
Top 5 Terms:
adjust : 0.2997
decisions : 0.2997
practice : 0.2997
successful : 0.2997
strong : 0.2997

Document: Italian Pasta
Top 5 Terms:
different : 0.6211
quality : 0.3105
long : 0.3105
tomatoes : 0.3105
improve : 0.3105

Document: Space Exploration
Top 5 Terms:
medical : 0.5015
international : 0.5015
health : 0.4388
data : 0.3901
water : 0.3901

Document: Cybersecurity
Top 5 Terms:
systems : 0.4348
data : 0.3382
strong : 0.2174
software : 0.2174
training : 0.2174

Document: Music Theory
Top 5 Terms:
practice : 0.4994
personal : 0.2497
live : 0.2497
software : 0.2497
training : 0.2497

Document: Gardening Vegetables
Top 5 Terms:
improves : 0.4361
successful : 0.218
quality : 0.218
tomatoes : 0.218
vegetables : 0.218

In [ ]:
# Display terms containing two words
bigrams = [
    term for term in new_terms
    if " " in term
]

print("Total Bigrams:", len(bigrams))
print("\nSample Bigrams:")

for bigram in bigrams[:20]:
    print(bigram)

Total Bigrams: 3

Sample Bigrams:
economic growth
policy involves
trade offs


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
import string

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# Get English stop words
stop_words = set(stopwords.words('english'))

# Combine all documents
all_text = " ".join(documents).lower()

# Tokenize text
words = word_tokenize(all_text)

# Count stop words
stop_word_list = [
    word for word in words
    if word in stop_words
]

# Calculate frequency
stop_word_frequency = Counter(stop_word_list)

# Print results
print("Total stop-word occurrences:", len(stop_word_list))
print("\nStop-word frequencies:")

for word, frequency in stop_word_frequency.most_common():
    print(word, ":", frequency)

Total stop-word occurrences: 199

Stop-word frequencies:
and : 76
to : 12
can : 8
on : 8
a : 8
are : 6
for : 6
the : 6
is : 5
of : 5
by : 4
from : 4
as : 4
in : 4
with : 4
that : 3
more : 3
or : 3
than : 2
between : 2
be : 2
through : 2
at : 2
over : 2
before : 1
if : 1
them : 1
such : 1
but : 1
while : 1
now : 1
about : 1
should : 1
until : 1
then : 1
only : 1
has : 1
so : 1
both : 1
each : 1
after : 1
how : 1


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
!pip install pytest

In [ ]:
%cd /content/unstructuredata_tfidf_repo

/content/unstructuredata_tfidf_repo


In [ ]:
!ls
!find . -name "test_*.py"

data	 notebooks  requirements.txt  task_to_do.txt
LICENSE  README.md  src		      tests
./tests/test_tfidf_utils.py


In [ ]:
!pytest -q

...                                                                      [100%]
3 passed in 2.81s


In [1]:
import os

if not os.path.exists("/content/unstructuredata_tfidf_repo"):
    !git clone https://github.com/mateenyaqoob/unstructuredata_tfidf_repo.git

%cd /content/unstructuredata_tfidf_repo

Cloning into 'unstructuredata_tfidf_repo'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 84 (delta 28), reused 59 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 35.77 KiB | 7.15 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/unstructuredata_tfidf_repo


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load documents if they are not already loaded
from src.load_data import load_documents
titles, documents = load_documents()

# Create TF-IDF matrix
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(documents)

# Define search function
def search_documents(query):
    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector, tfidf_matrix
    ).flatten()

    ranked_indices = scores.argsort()[::-1]

    print("\nQuery:", query)
    print("-" * 40)

    for i in ranked_indices:
        print(titles[i], ":", round(scores[i], 4))

    return titles[ranked_indices[0]]

# Run the three queries
result1 = search_documents(
    "renewable energy and climate policy"
)
result2 = search_documents(
    "password security and phishing"
)
result3 = search_documents(
    "healthy sleep exercise and diet"
)

# Check expected results
print("\n--- Validation ---")

expected = [
    "Climate Change",
    "Cybersecurity",
    "Public Health"
]

actual = [result1, result2, result3]

for exp, got in zip(expected, actual):
    print("Expected:", exp)
    print("Actual:", got)
    print("Correct:", exp == got)
    print()


Query: renewable energy and climate policy
----------------------------------------
Climate Change : 0.3235
Economics Inflation : 0.0952
Gardening Vegetables : 0.0485
Public Health : 0.0
Cybersecurity : 0.0
Music Theory : 0.0
Space Exploration : 0.0
Italian Pasta : 0.0
Basketball Strategy : 0.0
AI in Healthcare : 0.0

Query: password security and phishing
----------------------------------------
Cybersecurity : 0.1764
Public Health : 0.0
Economics Inflation : 0.0
Gardening Vegetables : 0.0
Music Theory : 0.0
Space Exploration : 0.0
Italian Pasta : 0.0
Basketball Strategy : 0.0
Climate Change : 0.0
AI in Healthcare : 0.0

Query: healthy sleep exercise and diet
----------------------------------------
Public Health : 0.2849
Gardening Vegetables : 0.0447
Economics Inflation : 0.0
Music Theory : 0.0
Cybersecurity : 0.0
Space Exploration : 0.0
Italian Pasta : 0.0
Basketball Strategy : 0.0
Climate Change : 0.0
AI in Healthcare : 0.0

--- Validation ---
Expected: Climate Change
Actual: Clima